# 02 - Prepare the held-out test events

**Settings:** Accelerator = **None (CPU)**, Internet = On. Run with **Save Version > Save & Run All (Commit)**,
not the editor's Run All: a committed run keeps going when the laptop is closed and its output is saved.

**Inputs** (Add Input > Your Work > Notebooks):
none

**Output:** `ufm-test` (~3.3 GB)  |  **Approx. time:** 0.5-1 h

Downloads the test chips of Weihui (China), Nova Kakhovka (Ukraine) and Jubba (Somalia, two image pairs) plus their full-frame reference maps, and adds the auxiliary layers. Expected chips: 2475 / 2893 / 1779 / 1032 (997 empty Jubba_2 files on the server are skipped).

In [ ]:
# ---- Settings -------------------------------------------------------------
GITHUB_REPO = "https://github.com/mhdnasim6-boop/urban-flood-mapping.git"
BRANCH = "main"
# ---------------------------------------------------------------------------
# Private repo: add a Kaggle secret named GITHUB_TOKEN (Add-ons > Secrets).
# Optional: a secret named HF_TOKEN (Hugging Face read token) avoids download rate limits.
import glob, os, subprocess
from pathlib import Path

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

url = GITHUB_REPO
if secret("GITHUB_TOKEN"):
    url = url.replace("https://", f"https://{secret('GITHUB_TOKEN')}@")
if secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = secret("HF_TOKEN")
subprocess.run(["rm", "-rf", "/tmp/ufm"])
subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, "/tmp/ufm"], check=True)
os.chdir("/tmp/ufm")
print("code version:", subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

def find_input(name, notebook):
    """Locate an attached input folder or stop immediately with a clear message."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    if not hits:
        raise SystemExit(f"STOP: input '{name}' not found. Add the output of notebook {notebook} "
                         "(Add Input > Your Work > Notebooks) and run again.")
    print(f"{name}: {hits[0]}")
    return hits[0]

In [ ]:
!pip install -q rasterio 2>/dev/null; python -c "import rasterio; print('rasterio', rasterio.__version__)"

In [ ]:
!python scripts/prepare_test.py --out /kaggle/working/ufm-test --threads 16

In [ ]:
!python scripts/build_aux.py --root /kaggle/working/ufm-test --cache /tmp/tiles --workers 4 --max-cache-gb 6

In [ ]:
!du -sh /kaggle/working/ufm-test/*
!cat /kaggle/working/ufm-test/event_stats.json